In [1]:
import sys
sys.path.append("..")
import os
import torch
import yaml
from pathlib import Path
from datasets import DatasetDict
from datasets import load_from_disk

from src.data import PROJECT_ROOT, evaluate_model_capability
from src.dataset_cot import prepare_cot_dataset, get_cot_prompt_template, evaluate_cot_capability
from src.llm_upgrade import (
    prepare_model_for_finetune,
    train_lora_model,
    save_finetuned_model,
    wrap_for_transformer_lens
)

In [2]:
with open(PROJECT_ROOT / "configs" / "finetune.yaml", "r") as f:
    config = yaml.safe_load(f)

In [3]:
VARIANT = config.get("variant", "depth-0")
USE_SMALL = config.get("use_small", True)
cache_suffix = f"{VARIANT}_small" if USE_SMALL else VARIANT
cache_path = PROJECT_ROOT / "data" / "processed" / f"ruletaker_{cache_suffix}"
raw_dataset = load_from_disk(str(cache_path))

# Создание CoT датасета

In [4]:
# Применяем CoT-форматирование
template = get_cot_prompt_template(model_name=config.get("model_name", "default"))
cot_train = prepare_cot_dataset(raw_dataset["train"], template=template, depth=VARIANT)
cot_eval = prepare_cot_dataset(raw_dataset["dev"], template=template, depth=VARIANT) if "dev" in raw_dataset else None

In [5]:
cot_train

Dataset({
    features: ['id', 'text', 'label'],
    num_rows: 2000
})

In [6]:
cot_eval

Dataset({
    features: ['id', 'text', 'label'],
    num_rows: 500
})

In [7]:
print(cot_eval[499])

{'id': 'AttNoneg-D0-6417_2', 'text': "Theory: Charlie is smart. Erin is quiet. Gary is nice. If Erin is smart then Erin is nice. If someone is cold and young then they are smart. If someone is young and green then they are cold.\nAssertion: Gary is not nice.\nReasoning: Let's think step by step. The assertion contradicts the given facts.\nAnswer: False", 'label': False}


In [8]:
cot_dir = PROJECT_ROOT / "data" / "processed" / f"ruletaker_{VARIANT}_cot"
if cot_eval is not None:
    DatasetDict({"train": cot_train, "dev": cot_eval}).save_to_disk(str(cot_dir))
else:
    cot_train.save_to_disk(str(cot_dir))

Saving the dataset (0/1 shards):   0%|          | 0/2000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

# Дообучение (LoRA)

In [5]:
# Подготовка модели к файнтюнингу
MODEL_NAME = config.get("model_name", "Qwen/Qwen2.5-1.5B")
USE_QLORA = config.get("use_qlora", True)

In [10]:
model, tokenizer = prepare_model_for_finetune(
    MODEL_NAME,
    use_qlora=USE_QLORA,
    device="cuda"
)

`torch_dtype` is deprecated! Use `dtype` instead!


trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


In [11]:
trained_model, metrics = train_lora_model(
    model=model,
    tokenizer=tokenizer,
    train_dataset=cot_train,
    eval_dataset=cot_eval,
    config=config,
    max_length=config.get("max_length", 512)
)

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
c:\MyPythonProjects\mephi_diss\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
50,0.686800,0.662689
100,0.612000,0.629696
150,0.573100,0.629342
200,0.539800,0.641311
250,0.493500,0.676573
300,0.404200,0.771059
350,0.360800,0.813057


c:\MyPythonProjects\mephi_diss\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\MyPythonProjects\mephi_diss\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\MyPythonProjects\meph

In [4]:
adapter_path = PROJECT_ROOT / "results/checkpoints/finetune/checkpoint-150"

In [ ]:
save_finetuned_model(trained_model, tokenizer, str(adapter_path))

NameError: name 'trained_model' is not defined

# Тест CoT дообученной (LoRA) модели

In [5]:
hooked_model, _ = wrap_for_transformer_lens(
    base_model_name="Qwen/Qwen2.5-1.5B",
    adapter_path=str(adapter_path),
    device="cuda"
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model Qwen/Qwen2.5-1.5B into HookedTransformer


In [6]:
hooked_model.cfg.n_layers

28

In [7]:
full_data_path = PROJECT_ROOT / "data" / "processed" / f"ruletaker_{VARIANT}"
full_dataset = load_from_disk(str(full_data_path))

In [8]:
TEST_SUBSET_SIZE = 300
eval_subset = full_dataset["test"].shuffle(seed=42).select(range(TEST_SUBSET_SIZE))

In [9]:
eval_prefix = template.replace("{reasoning_steps}", "").replace("{label}", "").rstrip()
if not eval_prefix.endswith("Answer: "):
    eval_prefix += " Answer: "

In [12]:
acc = evaluate_cot_capability(
    model=hooked_model,
    dataset=eval_subset,
    prompt_template=eval_prefix,
    batch_size=16,
    device="cuda"
)

Evaluating CoT capability: 100%|██████████| 19/19 [02:10<00:00,  6.85s/it]


In [13]:
acc

0.6633333333333333

# Тест прямого формата завершения

In [9]:
direct_prefix = """Theory: {theory}
Assertion: {assertion}
Answer: """

In [10]:
acc = evaluate_cot_capability(
    model=hooked_model,
    dataset=eval_subset,
    prompt_template=direct_prefix,
    batch_size=32,
    device="cuda"
)

Evaluating CoT capability:   0%|          | 0/10 [00:00<?, ?it/s]

Evaluating CoT capability: 100%|██████████| 10/10 [01:58<00:00, 11.89s/it]


In [11]:
acc

0.6866666666666666

In [13]:
!nvidia-smi

Thu Apr  9 23:19:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.59                 Driver Version: 591.59         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   59C    P8              9W /  115W |    5243MiB /   6144MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----